## WSW Gates - Beach Courses

Generate beach courses for Weymouth Speed Week

In [1]:
import os
import sys

import jinja2

import pyproj

### SWCP Waypoints

South West Coast Path waypoints identified on Google Earth

In [2]:
# OTC
waypoint1 = (50.57482500, -2.46500000)

# Billy Winters
waypoint2 = (50.57890556, -2.46826944)

### Jinja Setup

Prepare environment for Jinja templates

In [3]:
projdir = os.path.realpath(os.path.join(sys.path[0], '..'))

coursesPath = os.path.join(projdir, 'courses')

gtxPath = os.path.join(coursesPath, 'gtx')
kmlPath = os.path.join(coursesPath, 'kml')

templateLoader = jinja2.FileSystemLoader([gtxPath, kmlPath])

templateEnv = jinja2.Environment(loader=templateLoader,
                                 autoescape=True, trim_blocks=True, lstrip_blocks=True)

### Calculate distance and azimuths using pyproj

The pyproj library returns distances in metres, and azimuths betwen -180 and +180

In [4]:
geod = pyproj.Geod(ellps='WGS84')

In [5]:
# Use SWCP waypoints
lat1, lon1 = waypoint1
lat2, lon2 = waypoint2

# Determine forward and back azimuths, plus distance between the waypoints
forward_azimuth, back_azimuth, distance = geod.inv(lon1, lat1, lon2, lat2)

# Convert negative values to positive values
forward_azimuth = (forward_azimuth + 360) % 360
back_azimuth = (back_azimuth + 360) % 360
line_azimuth = (forward_azimuth + 90)  % 360

# Report the results
print(f"Distance: {distance:.3f} meters")
print(f"Heading: {forward_azimuth:.3f} degrees")

Distance: 509.587 meters
Heading: 332.971 degrees


### Calculate Points for Start / Finish Line

Simple forward projection

In [6]:
def calculatePoints(waypoints, i, line_azimuth, gate_width):
    '''Get corners'''

    c1_lon = waypoints.lons[i]
    c1_lat = waypoints.lats[i]
    c2_lon, c2_lat, ignore = geod.fwd(c1_lon, c1_lat, line_azimuth, gate_width)
    mid_lon, mid_lat, ignore = geod.fwd(c1_lon, c1_lat, line_azimuth, gate_width / 2)

    return c1_lon, c1_lat, c2_lon, c2_lat, mid_lon, mid_lat

### Generate Gate File

Use Jinja to generate .gtx file from template

In [7]:
def saveGtx(w1, w2, i, line_azimuth):
    '''Save GTX for GPSResults'''
    
    track_length = -500
    gate_width = 1500

    # Calculate corners, and start + end points
    
    c1_lon, c1_lat, c2_lon, c2_lat, start_lon, start_lat = calculatePoints(w1, i, line_azimuth, gate_width)   
    c3_lon, c3_lat, c4_lon, c4_lat, finish_lon, finish_lat = calculatePoints(w2, i, line_azimuth, gate_width)
    
    # Save Gate XML
    
    corners_lat_lon = "{:.7f} {:.7f} {:.7f} {:.7f} {:.7f} {:.7f} {:.7f} {:.7f}".format(
        c1_lat, c1_lon, c2_lat, c2_lon, c3_lat, c3_lon, c4_lat, c4_lon)
    
    template = templateEnv.get_template("template.gtx")
    gtx = template.render(track_length=track_length, gate_width=gate_width,
                          start_lat=round(start_lat, 7), start_lon=round(start_lon, 7),
                          finish_lon=round(finish_lon, 7), finish_lat=round(finish_lat, 7),
                          corners_lat_lon=corners_lat_lon
                         )
    
    gtxFile = os.path.join(gtxPath, 'test.gtx')
    with open(gtxFile, 'w', encoding='utf-8') as f:
    	f.write(gtx)

### Generate KML File

Use Jinja to generate .kml file from template

In [8]:
def saveKml(w1, w2, i, line_azimuth):
    '''Save KML for Google Earth'''
    
    gate_width = 1500

    # New approach

    mid_lon = (w1.lons[i] + w2.lons[i]) / 2
    mid_lat = (w1.lats[i] + w2.lats[i]) / 2

    azimuth = 325
    distance = 250

    # c1 = start line (shore), c3 = finish line (shore)
    c1_lon, c1_lat, ignore = geod.fwd(mid_lon, mid_lat, (azimuth + 180) % 360, distance)
    c3_lon, c3_lat, ignore = geod.fwd(mid_lon, mid_lat, azimuth, distance)

    # Move c1 + c3 towards chesil by 25 meters
    c1_lon, c1_lat, ignore = geod.fwd(c1_lon, c1_lat, (azimuth - 90) % 360, 25)
    c3_lon, c3_lat, ignore = geod.fwd(c3_lon, c3_lat, (azimuth - 90) % 360, 25)

    # c2 = start line (harbour), c4 = finish line (harbour)
    c2_lon, c2_lat, ignore = geod.fwd(c1_lon, c1_lat, (azimuth + 90) % 360, gate_width)
    c4_lon, c4_lat, ignore = geod.fwd(c3_lon, c3_lat, (azimuth + 90) % 360, gate_width)

    # Pointsat the middle of the start and finish lines
    start_lon, start_lat, ignore = geod.fwd(c1_lon, c1_lat, (azimuth + 90) % 360, gate_width / 2)
    finish_lon, finish_lat, ignore = geod.fwd(c3_lon, c3_lat, (azimuth + 90) % 360, gate_width / 2)
    
    # Calculate corners, and start + end points
    
    #c1_lon, c1_lat, c2_lon, c2_lat, start_lon, start_lat = calculatePoints(w1, i, line_azimuth, gate_width)   
    #c3_lon, c3_lat, c4_lon, c4_lat, finish_lon, finish_lat = calculatePoints(w2, i, line_azimuth, gate_width)

    # Save KML
    
    name = 'Weymouth Speed Week'
    
    polygon_coordinates = \
        "{:.7f},{:.7f},0 {:.7f},{:.7f},0 {:.7f},{:.7f},0 {:.7f},{:.7f},0 {:.7f},{:.7f},0".format(
        c1_lon, c1_lat, c3_lon, c3_lat, c4_lon, c4_lat, c2_lon, c2_lat, c1_lon, c1_lat)
    
    template = templateEnv.get_template("template.kml")
    kml = template.render(name=name,
                          start_lat=round(start_lat, 7), start_lon=round(start_lon, 7),
                          finish_lon=round(finish_lon, 7), finish_lat=round(finish_lat, 7),
                          polygon_coordinates=polygon_coordinates
                         )
    
    kmlFile = os.path.join(kmlPath, 'test.kml')
    with open(kmlFile, 'w', encoding='utf-8') as f:
    	f.write(kml)

### Generate Multiple Courses

Typically 50 meter intervals

In [9]:
interval = 50

w1 = geod.fwd_intermediate(lon1, lat1, back_azimuth, npts=6, del_s=interval, initial_idx=0,
                           return_back_azimuth=True)
w2 = geod.fwd_intermediate(lon2, lat2, back_azimuth, npts=6, del_s=interval, initial_idx=0,
                           return_back_azimuth=True)

In [10]:
i = 3
saveGtx(w1, w2, i, line_azimuth)
saveKml(w1, w2, i, line_azimuth)